In [1]:
import pandas as pd
import numpy as np
import os
import joblib

TEST_DIR = "test_data"
DATA_DIR = "data"

print("Preparing test dataset...")

# Load trained artifacts
feature_names = joblib.load(os.path.join(DATA_DIR, "feature_names.save"))
scaler        = joblib.load(os.path.join(DATA_DIR, "scaler.save"))

# Load CSV(s)
files = [f for f in os.listdir(TEST_DIR) if f.endswith(".csv")]

if not files:
    print("No CSV found in test_data/")
    exit()

df_list = []

for f in files:
    print("Reading:", f)
    df = pd.read_csv(os.path.join(TEST_DIR, f))
    df.columns = df.columns.str.strip()
    df_list.append(df)

df = pd.concat(df_list, ignore_index=True)

# Save original processed flows (VERY IMPORTANT)
df.to_csv(os.path.join(TEST_DIR, "processed_flows.csv"), index=False)

# Drop label column if exists
if "Label" in df.columns:
    df = df.drop(columns=["Label"])

# Match training features
for col in feature_names:
    if col not in df.columns:
        df[col] = 0

df = df[feature_names]

# Clean data
df = df.replace([np.inf, -np.inf], np.nan)
df = df.fillna(df.median())

# Convert + scale
X = df.astype(np.float32)
X_scaled = scaler.transform(X)

# Save test data (ONLY in test_data)
np.save(os.path.join(TEST_DIR, "X_test.npy"), X_scaled)

# Dummy labels (needed to run detect.py)
y_dummy = np.zeros(len(X_scaled), dtype=np.int32)
np.save(os.path.join(TEST_DIR, "y_test.npy"), y_dummy)

print("Test dataset ready. Now run detect.py")

Preparing test dataset...
Reading: log5.pcap_Flow.csv
Test dataset ready. Now run detect.py
